# PMC Article Labeling Pipeline
This notebook clones the project repo, installs dependencies, verifies the vLLM server, and runs the labeling loop.

> **Before running:** Make sure your vLLM server is accessible from Colab (e.g. via ngrok tunnel or a public URL) and fill in the configuration cell below.

## 1 — Clone the repository

In [ ]:
import os

REPO_URL   = "https://github.com/EasternPharma/PmcArticleLabel.git"
REPO_DIR   = "PmcArticleLabel"

if os.path.isdir(REPO_DIR):
    print(f"Repo already exists — pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    print(f"Cloning {REPO_URL} ...")
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 2 — Install dependencies

In [ ]:
import sys
sys.path.insert(0, os.getcwd())

from CheckLibraries import main as check_libraries

assert check_libraries(), "Some libraries failed to install. See output above."

## 3 — Configuration
Set your server URLs and model name here.

In [ ]:
# Article management server
API_BASE_URL       = "http://localhost:8000"   # e.g. https://xxxx.ngrok.io

# vLLM inference server
VLLM_BASE_URL      = "http://localhost:8001"   # e.g. https://yyyy.ngrok.io
MODEL_NAME         = "Qwen/Qwen3.5-4B"

# Labeling batch size and polling interval (seconds)
BATCH_SIZE         = 100
POLL_INTERVAL      = 5

## 4 — Verify vLLM server

In [ ]:
from CheckVLLM import main as check_vllm

assert check_vllm(vllm_base_url=VLLM_BASE_URL, model_name=MODEL_NAME), \
    "vLLM check failed. Make sure the server is running and reachable."

## 5 — Run the labeling pipeline
Runs continuously. **Interrupt the kernel** (Runtime → Interrupt) to stop.

In [ ]:
import time
from tqdm.notebook import tqdm
from ApiCall import ApiCall
from ArticleLabelHelper import ArticleLabelHelper

api_client   = ApiCall(base_url=API_BASE_URL)
label_helper = ArticleLabelHelper(vllm_base_url=VLLM_BASE_URL, model_name=MODEL_NAME)

total_batches  = 0
total_labeled  = 0

print("Starting PMC article labeling loop. Interrupt the kernel to stop.\n")

with tqdm(desc="Batches processed", unit="batch") as batch_bar:
    while True:
        try:
            articles = api_client.get_unlabeled_articles(batch_size=BATCH_SIZE)
            if not articles:
                batch_bar.set_postfix_str("waiting for articles...")
                time.sleep(POLL_INTERVAL)
                continue

            results = label_helper.label_batch(articles)
            if not results:
                print("[pipeline] No results produced from labeling.")
                continue

            success = api_client.update_article_labels(results)
            if success:
                total_batches += 1
                total_labeled += len(results)
                batch_bar.update(1)
                batch_bar.set_postfix_str(f"total labeled: {total_labeled}")

        except KeyboardInterrupt:
            print(f"\n[pipeline] Stopped by user.")
            print(f"[pipeline] Total batches: {total_batches} | Total articles labeled: {total_labeled}")
            break
        except Exception as e:
            print(f"[pipeline] Unexpected error: {e}")
            time.sleep(POLL_INTERVAL)